# Upload Papers + READMEs to HuggingFace Hub

Upload or update the **paper.md** and **README.md** (model card) for each HuggingFace model repo.
**Run locally** — reads papers from `docs/` in the git repo.

| What | Notebook |
|------|----------|
| Upload model weights | `upload_models_to_hf.ipynb` (Databricks or local) |
| Upload/update papers + READMEs | **This notebook** |

## When to use this notebook
- First time: after uploading model weights, run this to add papers + READMEs
- Paper revision: edit `docs/mouse_paper_draft.md` or `docs/human_paper_draft.md`, then re-run
- README update: edit the model card templates in Cell 1, then re-run

HuggingFace repos are git-backed — every update is versioned. You can re-run freely.

## Usage
```bash
cd histological-image-analysis
export HUGGING_FACE_TOKEN=hf_your_token_here
jupyter notebook notebooks/upload_papers_to_hf.ipynb
```

In [1]:
# Cell 0 — Configuration

import os
from pathlib import Path
from dotenv import load_dotenv

# --- Load .env file (looks in repo root) ---
for _env_candidate in [Path(".env"), Path("../.env")]:
    if _env_candidate.exists():
        load_dotenv(_env_candidate)
        print(f"Loaded .env from {_env_candidate.resolve()}")
        break

# --- Resolve docs directory ---
_doc_candidates = [Path("docs"), Path("../docs")]
DOCS_DIR = next((p for p in _doc_candidates if p.exists()), None)

if not DOCS_DIR:
    raise FileNotFoundError(
        "docs/ directory not found. Run this notebook from the repo root "
        "or the notebooks/ directory."
    )

MOUSE_PAPER = DOCS_DIR / "mouse_paper_draft.md"
HUMAN_PAPER = DOCS_DIR / "human_paper_draft.md"

HF_TOKEN = os.environ.get("HUGGING_FACE_TOKEN")
HF_USERNAME = "Noel-Niko"
MODEL_BASE = "dinov2-upernet-20260322-histology-annotation"

print(f"Docs dir:     {DOCS_DIR.resolve()}")
print(f"Mouse paper:  {'FOUND' if MOUSE_PAPER.exists() else 'MISSING'} ({MOUSE_PAPER})")
print(f"Human paper:  {'FOUND' if HUMAN_PAPER.exists() else 'MISSING'} ({HUMAN_PAPER})")
print(f"HF token:     {'found (' + HF_TOKEN[:8] + '...)' if HF_TOKEN else 'NOT SET'}")

if not HF_TOKEN:
    print("\nAdd HUGGING_FACE_TOKEN=hf_... to .env or: export HUGGING_FACE_TOKEN=hf_...")

Loaded .env from /Users/xnxn040/PycharmProjects/Personal-Projects/histological-image-analysis/.env
Docs dir:     /Users/xnxn040/PycharmProjects/Personal-Projects/histological-image-analysis/docs
Mouse paper:  FOUND (../docs/mouse_paper_draft.md)
Human paper:  FOUND (../docs/human_paper_draft.md)
HF token:     found (hf_FnOaZ...)


In [ ]:
# Cell 1 — Define model card READMEs and paper mappings
#
# Edit these templates to update the HuggingFace model cards.
# Then re-run Cell 2 to push changes.

REPOS = {
    "mouse": {
        "repo_id": f"{HF_USERNAME}/{MODEL_BASE}-mouse",
        "paper_path": MOUSE_PAPER,
        "readme": """---
license: apache-2.0
tags:
  - brain-segmentation
  - histology
  - dinov2
  - upernet
  - neuroscience
  - allen-brain-institute
---

# Brain Region Segmentation — Mouse Brain

DINOv2-Large + UperNet model fine-tuned for semantic segmentation of
mouse brain structures in Nissl-stained histological sections.

## Model Details

| Attribute | Value |
|-----------|-------|
| Architecture | DINOv2-Large (304M) + UperNet (38M) |
| Classes | 1,328 |
| Input Size | 518x518 |
| Training Data | Allen Brain Institute CCFv3 10um Nissl staining |
| mIoU (center-crop) | 74.8% |
| mIoU (sliding window) | 79.1% |

## Usage

```bash
git clone https://github.com/Noel-Niko/histological-image-analysis
cd histological-image-analysis
make install
make download-models-mouse
make annotate-mouse IMAGES=/path/to/your/slides/
```

## Paper

**Transfer Learning for Ultra-Fine-Grained Brain Region Segmentation: An Ablation Study with DINOv2 + UperNet on 1,328 Allen Mouse Brain Atlas Structures**

We conduct a systematic ablation study across 9 training runs on pixel-level segmentation of 1,328 brain structures from the Allen Mouse Brain Atlas CCFv3. The two most impactful interventions are backbone partial fine-tuning (+8.5% mIoU) and extended training from 100 to 200 epochs (+6.0% mIoU). Three attempted improvements — weighted Dice+CE loss, aggressive augmentation, and test-time augmentation — all degrade performance.

See [`paper.md`](paper.md) in this repo for the full paper.

## Citation

If you use this model, please cite the training data sources and the paper included in this repository.

## Repository

Full source code, training notebooks, and all models: https://github.com/Noel-Niko/histological-image-analysis

## Maintaining This Repo

To update model weights, papers, or this README:

```bash
cd histological-image-analysis
export HUGGING_FACE_TOKEN=hf_your_token_here

# Update model weights (Databricks or local):
jupyter notebook notebooks/upload_models_to_hf.ipynb

# Update papers + READMEs (local only):
jupyter notebook notebooks/upload_papers_to_hf.ipynb
```
""",
    },
    "human": {
        "repo_id": f"{HF_USERNAME}/{MODEL_BASE}-human",
        "paper_path": HUMAN_PAPER,
        "readme": """---
license: apache-2.0
tags:
  - brain-segmentation
  - histology
  - dinov2
  - upernet
  - neuroscience
  - allen-brain-institute
---

# Brain Region Segmentation — Human Brain (Allen Depth-3)

DINOv2-Large + UperNet model fine-tuned for semantic segmentation of
human brain regions in Nissl-stained histological sections.

## Model Details

| Attribute | Value |
|-----------|-------|
| Architecture | DINOv2-Large (304M) + UperNet (38M) |
| Classes | 44 (depth-3 brain regions) |
| Input Size | 518x518 |
| Training Data | Allen Human Brain Atlas (6 donors, Nissl staining) |
| mIoU (val center-crop) | 65.5% |
| mIoU (test sliding window) | 65.0% |
| Pixel accuracy (test) | 99.1% |

## Usage

```bash
git clone https://github.com/Noel-Niko/histological-image-analysis
cd histological-image-analysis
make install
make download-models-human-allen
make annotate-human-allen IMAGES=/path/to/your/slides/
```

## Paper

**Cross-Species Transfer of Ultra-Fine-Grained Brain Segmentation: From Mouse to Human with DINOv2 + UperNet**

We extend the DINOv2-Large + UperNet approach from mouse (1,328 classes, 79.1% mIoU) to human brain tissue using the Allen Human Brain Atlas (sparse SVG annotations, 6 donors). The depth-3 model (44 brain regions) achieves 65.5% val CC mIoU and 65.0% test SW mIoU with 99.1% pixel accuracy. Major structures (cerebellum, cerebral cortex, thalamus, pons) exceed 99% IoU.

See [`paper.md`](paper.md) in this repo for the full paper.

## Citation

If you use this model, please cite the training data sources and the paper included in this repository.

## Repository

Full source code, training notebooks, and all models: https://github.com/Noel-Niko/histological-image-analysis

## Maintaining This Repo

To update model weights, papers, or this README:

```bash
cd histological-image-analysis
export HUGGING_FACE_TOKEN=hf_your_token_here

# Update model weights (Databricks or local):
jupyter notebook notebooks/upload_models_to_hf.ipynb

# Update papers + READMEs (local only):
jupyter notebook notebooks/upload_papers_to_hf.ipynb
```
""",
    },
    "human-bigbrain": {
        "repo_id": f"{HF_USERNAME}/{MODEL_BASE}-human-bigbrain",
        "paper_path": HUMAN_PAPER,
        "readme": """---
license: apache-2.0
tags:
  - brain-segmentation
  - histology
  - dinov2
  - upernet
  - neuroscience
  - bigbrain
---

# Brain Region Segmentation — Human Brain (BigBrain Tissue Classification)

DINOv2-Large + UperNet model fine-tuned for semantic segmentation of
human brain tissue types in histological sections.

## Model Details

| Attribute | Value |
|-----------|-------|
| Architecture | DINOv2-Large (304M) + UperNet (38M) |
| Classes | 10 (tissue types) |
| Input Size | 518x518 |
| Training Data | BigBrain 3D histological volume (200um, 9-class tissue classification) |
| mIoU (center-crop) | 60.8% |
| mIoU (sliding window) | 61.3% |

## Tissue Classes

| ID | Class |
|----|-------|
| 0 | Background |
| 1 | Gray Matter |
| 2 | White Matter |
| 3 | Cerebrospinal Fluid |
| 4 | Meninges |
| 5 | Blood Vessels |
| 6 | Bone/Skull |
| 7 | Muscle |
| 8 | Artifact |
| 9 | Other/Unknown |

## Usage

```bash
git clone https://github.com/Noel-Niko/histological-image-analysis
cd histological-image-analysis
make install
make download-models-human-bigbrain
make annotate-human-bigbrain IMAGES=/path/to/your/slides/
```

## Paper

**Cross-Species Transfer of Ultra-Fine-Grained Brain Segmentation: From Mouse to Human with DINOv2 + UperNet**

This model is Track B of a three-track human brain segmentation study. It uses the BigBrain 200um classified volume with dense 9-class tissue annotations (Merker stain). The BigBrain model serves as a tissue type classifier — complementary to the Allen depth-3 model's role as a brain region identifier.

See [`paper.md`](paper.md) in this repo for the full paper.

## Citation

If you use this model, please cite the training data sources and the paper included in this repository.

## Repository

Full source code, training notebooks, and all models: https://github.com/Noel-Niko/histological-image-analysis

## Maintaining This Repo

To update model weights, papers, or this README:

```bash
cd histological-image-analysis
export HUGGING_FACE_TOKEN=hf_your_token_here

# Update model weights (Databricks or local):
jupyter notebook notebooks/upload_models_to_hf.ipynb

# Update papers + READMEs (local only):
jupyter notebook notebooks/upload_papers_to_hf.ipynb
```
""",
    },
}

# Which repos to update (change to a list of specific keys, or "all")
UPDATE_REPOS = "all"  # Options: "all", "mouse", "human", "human-bigbrain"

if UPDATE_REPOS == "all":
    targets = list(REPOS.keys())
else:
    targets = [UPDATE_REPOS]

print(f"\nWill update: {targets}")

In [3]:
# Cell 2 — Upload papers + READMEs to HuggingFace

from huggingface_hub import HfApi, create_repo

if not HF_TOKEN:
    raise RuntimeError("HUGGING_FACE_TOKEN not set.")

api = HfApi(token=HF_TOKEN)

for species in targets:
    repo_cfg = REPOS[species]
    repo_id = repo_cfg["repo_id"]
    paper_path = repo_cfg["paper_path"]
    readme_content = repo_cfg["readme"]

    print(f"\n--- {species}: {repo_id} ---")

    # Ensure repo exists
    create_repo(repo_id=repo_id, token=HF_TOKEN, exist_ok=True)

    # Upload paper.md
    if paper_path.exists():
        paper_content = paper_path.read_text(encoding="utf-8")
        api.upload_file(
            path_or_fileobj=paper_content.encode("utf-8"),
            path_in_repo="paper.md",
            repo_id=repo_id,
            token=HF_TOKEN,
        )
        print(f"  Uploaded paper.md ({len(paper_content):,} chars from {paper_path.name})")
    else:
        print(f"  SKIP paper.md: {paper_path} not found")

    # Upload README.md (model card)
    api.upload_file(
        path_or_fileobj=readme_content.encode("utf-8"),
        path_in_repo="README.md",
        repo_id=repo_id,
        token=HF_TOKEN,
    )
    print(f"  Uploaded README.md ({len(readme_content):,} chars)")
    print(f"  View: https://huggingface.co/{repo_id}")

print("\nDone! Papers and READMEs updated on HuggingFace.")


--- mouse: Noel-Niko/dinov2-upernet-20260322-histology-annotation-mouse ---


/Users/xnxn040/PycharmProjects/Personal-Projects/histological-image-analysis/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


  Uploaded paper.md (36,555 chars from mouse_paper_draft.md)
  Uploaded README.md (2,083 chars)
  View: https://huggingface.co/Noel-Niko/dinov2-upernet-20260322-histology-annotation-mouse

--- human: Noel-Niko/dinov2-upernet-20260322-histology-annotation-human ---


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


  Uploaded paper.md (48,590 chars from human_paper_draft.md)
  Uploaded README.md (2,069 chars)
  View: https://huggingface.co/Noel-Niko/dinov2-upernet-20260322-histology-annotation-human

--- human-bigbrain: Noel-Niko/dinov2-upernet-20260322-histology-annotation-human-bigbrain ---


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


  Uploaded paper.md (48,590 chars from human_paper_draft.md)
  Uploaded README.md (2,222 chars)
  View: https://huggingface.co/Noel-Niko/dinov2-upernet-20260322-histology-annotation-human-bigbrain

Done! Papers and READMEs updated on HuggingFace.


In [4]:
# Cell 3 — Verify: list files in each repo

from huggingface_hub import list_repo_files

for species in targets:
    repo_id = REPOS[species]["repo_id"]
    print(f"\n{species}: {repo_id}")
    try:
        for f in sorted(list_repo_files(repo_id)):
            print(f"  {f}")
    except Exception as e:
        print(f"  ERROR: {e}")


mouse: Noel-Niko/dinov2-upernet-20260322-histology-annotation-mouse
  .gitattributes
  README.md
  config.json
  model.safetensors
  paper.md
  preprocessor_config.json
  training_args.bin

human: Noel-Niko/dinov2-upernet-20260322-histology-annotation-human
  .gitattributes
  README.md
  config.json
  model.safetensors
  paper.md
  preprocessor_config.json
  present_mapping.json
  training_args.bin

human-bigbrain: Noel-Niko/dinov2-upernet-20260322-histology-annotation-human-bigbrain
  .gitattributes
  README.md
  config.json
  model.safetensors
  paper.md
  preprocessor_config.json
  training_args.bin
